# 01 – Extract Payload from PCAP (Local)

Run this notebook on the **local machine** to extract fixed-length payload byte vectors
and packet-level metadata from raw `.pcap` / `.pcapng` files.

**Output** → `data/interim/payload_dataset/`
```
payload_256.npy   ← (N_packets, 256) uint8 payload matrix
metadata.csv      ← per-packet row (flow_id, timestamp, label, IPs, ports …)
stats.json        ← extraction statistics
flow_nodes.csv    ← flow node table   (when EXPORT_GRAPH_CSV=True)
packet_nodes.csv  ← packet node table (when EXPORT_GRAPH_CSV=True)
contain_edges.csv ← flow→packet edges  (when EXPORT_GRAPH_CSV=True)
link_edges.csv    ← packet→packet edges (when EXPORT_GRAPH_CSV=True)
```

**Next step**
- If student checkpoint **exists** → `02_export_student_embeddings.ipynb`
- If student checkpoint **does not exist** → upload `payload_256.npy` + `metadata.csv`
  to Kaggle and run `../kaggle/01_distill_student_cnn.ipynb` first.

In [ ]:
from pathlib import Path

# ── Input ──────────────────────────────────────────────────────────────────────
# One or more glob patterns to locate .pcap / .pcapng files.
PCAP_GLOBS = [
    "data/raw/**/*.pcap",
    "data/raw/**/*.pcapng",
]

# ── Output ────────────────────────────────────────────────────────────────────
OUTPUT_DIR = "data/interim/payload_dataset"

# ── Extraction hyperparameters ─────────────────────────────────────────────────
PAYLOAD_LENGTH           = 256    # fixed byte vector length (truncate + zero-pad)
FLOW_TIMEOUT_SECONDS     = 30.0   # idle seconds before a new flow starts
MAX_PACKETS_PER_FLOW     = 20     # graph fanout cap

# Use streaming mode to avoid holding the full dataset in RAM.
# Set False only for small datasets (< 1 M packets).
STREAM_TO_DISK           = True
NUM_WORKERS              = 0      # 0 = auto; 1 = sequential (safer on Windows)

# Export CSV tables for the flow / packet graph structure.
EXPORT_GRAPH_CSV         = False

In [14]:
# Verify the package is installed before proceeding.
import importlib, sys

try:
    importlib.import_module("graphslm_ids")
    print("graphslm_ids: OK")
except ImportError:
    print("Installing project package...")
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-e", ".", "--no-deps"])
    print("Done.")

graphslm_ids: OK


In [ ]:
import subprocess, sys
from pathlib import Path

# Tìm project root (thư mục chứa pyproject.toml)
_p = Path.cwd()
while _p != _p.parent:
    if (_p / "pyproject.toml").exists():
        break
    _p = _p.parent
PROJECT_ROOT = _p
print(f"Project root: {PROJECT_ROOT}")

cmd = [
    sys.executable, "-u", "-m",
    "graphslm_ids.offline_path.preprocessing.extract_payload_dataset",
    "--output-dir",                        OUTPUT_DIR,
    "--payload-length",                    str(PAYLOAD_LENGTH),
    "--graph-flow-timeout-seconds",        str(FLOW_TIMEOUT_SECONDS),
    "--graph-max-packets-per-flow",        str(MAX_PACKETS_PER_FLOW),
    "--num-workers",                       str(NUM_WORKERS),
    "--input-glob",                        *PCAP_GLOBS,
]

if STREAM_TO_DISK:
    cmd += ["--stream-to-disk", "--stream-mode", "single-pass"]

if EXPORT_GRAPH_CSV:
    cmd += ["--export-graph-csv"]

print("$", " ".join(cmd))
proc = subprocess.Popen(
    cmd, cwd=PROJECT_ROOT,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    encoding='utf-8', errors='replace', bufsize=1,
)
_prev_progress = False
for line in proc.stdout:
    text = line.rstrip('\n\r')
    is_progress = '%|' in text
    if is_progress:
        if _prev_progress:
            sys.stdout.write('\r' + text)
        else:
            sys.stdout.write(text)
    else:
        if _prev_progress:
            sys.stdout.write('\n')
        sys.stdout.write(text + '\n')
    sys.stdout.flush()
    _prev_progress = is_progress
if _prev_progress:
    sys.stdout.write('\n')
proc.wait()
if proc.returncode != 0:
    raise subprocess.CalledProcessError(proc.returncode, cmd)

In [18]:
# Verify outputs.
import json
import numpy as np
import pandas as pd
from pathlib import Path

out = PROJECT_ROOT / OUTPUT_DIR

payload = np.load(out / "payload_256.npy", mmap_mode="r")
print(f"payload_256.npy : {payload.shape}  dtype={payload.dtype}  "
      f"size={(out / 'payload_256.npy').stat().st_size / 1e9:.2f} GB")

meta = pd.read_csv(out / "metadata.csv")
print(f"metadata.csv    : {len(meta):,} rows")
print(f"  columns: {list(meta.columns)}")
if 'label' in meta.columns:
    print(f"  label distribution:\n{meta['label'].value_counts().to_string()}")

stats_path = out / "stats.json"
if stats_path.exists():
    print("\nstats.json:")
    print(json.dumps(json.loads(stats_path.read_text()), indent=2))

payload_256.npy : (5261944, 256)  dtype=uint8  size=1.35 GB
metadata.csv    : 5,261,944 rows
  columns: ['pcap_file', 'packet_index', 'timestamp', 'label', 'src_ip', 'dst_ip', 'src_port', 'dst_port', 'protocol', 'payload_len_raw']
  label distribution:
label
Benign                     1998702
VulnerabilityScan          1983424
Recon-OSScan                391769
Recon-HostDiscovery         388729
Recon-PortScan              281512
DDoS-ICMP_Fragmentation      59644
BrowserHijacking             35170
CommandInjection             34880
SqlInjection                 27794
XSS                          23000
Backdoor_Malware             19050
Recon-PingSweep              10897
Uploading_Attack              7373

stats.json:
{
  "created_at_utc": "2026-05-15T15:38:41.420241+00:00",
  "extraction_mode": "stream_to_disk_parallel_single_pass",
  "graph_csv": {
    "counts": {},
    "enabled": false,
    "flow_timeout_seconds": 30.0,
    "max_packets_per_flow": 20,
    "output_files": {},
    "rea